# IMDb Dataset Quick Inspection

This notebook is a lightweight **data sanity check**:
- Confirms the IMDb TSV files are found
- Loads only the necessary columns
- Applies the same title-type filtering used in the main analysis
- Merges `title.basics` + `title.ratings`
- Performs minimal cleaning so downstream notebooks/models don’t crash

> Required files (in the project root):  
> `title.basics.tsv` and `title.ratings.tsv` (or `.tsv.gz`)


## Dataset Description (what we use)

**title.basics.tsv**
- `tconst` (ID), `titleType`, `primaryTitle`, `startYear`, `genres`

**title.ratings.tsv**
- `tconst` (ID), `averageRating`, `numVotes`

We merge both tables on **`tconst`** to create the analysis dataset.


## Data Preprocessing (minimal)
We clean missing values, coerce numeric columns, filter relevant title types, merge on `tconst`, and create `logVotes` for stability.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

TARGET_TYPES = ['movie', 'tvSeries', 'tvMiniSeries', 'tvMovie', 'tvSpecial', 'video']

def resolve_imdb_path(filename: str) -> Path:
    """Find IMDb TSV/TSV.GZ whether you run from project root or /notebooks."""
    here = Path.cwd()
    candidates = [
        here / filename,
        here.parent / filename,
        Path(__file__).resolve().parent / filename if '__file__' in globals() else None,
        (Path(__file__).resolve().parent.parent / filename) if '__file__' in globals() else None,
    ]
    candidates = [c for c in candidates if c is not None]
    gz_candidates = [Path(str(c) + '.gz') for c in candidates]
    for p in candidates + gz_candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename} in: {candidates}")

basics_path = resolve_imdb_path('title.basics.tsv')
ratings_path = resolve_imdb_path('title.ratings.tsv')

print('Basics path:', basics_path)
print('Ratings path:', ratings_path)

BASICS_COLS = ['tconst', 'titleType', 'primaryTitle', 'startYear', 'genres']
RATINGS_COLS = ['tconst', 'averageRating', 'numVotes']

basics = pd.read_csv(
    basics_path,
    sep='\t',
    usecols=BASICS_COLS,
    low_memory=False,
    na_values=['\\N'],
    keep_default_na=True,
    on_bad_lines='skip'
)

ratings = pd.read_csv(
    ratings_path,
    sep='\t',
    usecols=RATINGS_COLS,
    low_memory=False,
    na_values=['\\N'],
    keep_default_na=True,
    on_bad_lines='skip'
)

print('Raw basics shape:', basics.shape)
print('Raw ratings shape:', ratings.shape)

# Filter to target types
basics = basics[basics['titleType'].isin(TARGET_TYPES)].copy()
basics['startYear'] = pd.to_numeric(basics['startYear'], errors='coerce')

# Merge
df = basics.merge(ratings, on='tconst', how='inner')

# Clean core numeric columns (prevents NaNs in modeling)
df['averageRating'] = pd.to_numeric(df['averageRating'], errors='coerce')
df['numVotes'] = pd.to_numeric(df['numVotes'], errors='coerce')
df = df.dropna(subset=['startYear', 'averageRating', 'numVotes']).copy()

df['startYear'] = df['startYear'].astype(int)
df['numVotes'] = df['numVotes'].astype(int)
df['genres'] = df['genres'].fillna('Unknown')

# Feature engineering used elsewhere
df['logVotes'] = np.log1p(df['numVotes'])

print('Prepared merged shape:', df.shape)
df.head()


Basics path: d:\University\Medtech\4th Year\Semester 2\CS434 - Data Analytics\Project\Data 3\imdb-movie-success-analysis-upgraded\title.basics.tsv
Ratings path: d:\University\Medtech\4th Year\Semester 2\CS434 - Data Analytics\Project\Data 3\imdb-movie-success-analysis-upgraded\title.ratings.tsv
Raw basics shape: (12256479, 5)
Raw ratings shape: (1631032, 3)
Prepared merged shape: (598680, 8)


,tconst,titleType,primaryTitle,startYear,genres,averageRating,numVotes,logVotes
0,tt0000009,movie,Miss Jerry,1894,Romance,5.3,232,5.451038
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897,"Documentary,News,Sport",5.3,584,6.371612
2,tt0000335,movie,Soldiers of the Cross,1900,"Biography,Drama",5.4,67,4.219508
3,tt0000502,movie,Bohemios,1905,Unknown,3.3,25,3.258097
4,tt0000574,movie,The Story of the Kelly Gang,1906,"Action,Adventure,Biography",6.0,1048,6.955593
